# Imports

In [1]:
import pandas as pd
import numpy as np
from langchain_openai import AzureChatOpenAI
import json
import re
import tqdm
from pathlib import Path
from datetime import datetime as dt

# EDA

1. Acceleration Pedal (Acc): 138 lines with labels [1, 0, 0, 0, 0, 0].

2. Wheel Steering Angle (WSA): 160 lines with labels [0, 1, 0, 0, 0, 0].

3. Wheel Speed (WS): 104 lines with labels [0, 0, 1, 0, 0, 0].

4. Yaw Rate (YR): 107 lines with labels [0, 0, 0, 1, 0, 0].

5. Steering Torque (ST): 118 lines with labels [0, 0, 0, 0, 1, 0].

6. Vehicle Speed (VS): 102 lines with labels [0, 0, 0, 0, 0, 1].

7. Acceleration Pedal and Wheel Steering Angle (Acc and WSA): 142 lines with labels

[1, 1, 0, 0, 0, 0].

8. Steering Angle and Wheel Speed (WSA and WS): 131 lines with labels [0, 1, 1, 0, 0, 0].

In [2]:
# read data and rename columns
df = pd.read_csv("data/requirements.csv")
df.columns = ["req", "s1", "s2", "s3", "s4", "s5", "s6"]

In [3]:
# make sure all columns except the first one are binary
for c in df.columns[1:]:
    assert df[c].drop_duplicates().shape[0] == 2

# Drop Sensor

In [4]:
# drop vihecal speed sensor
df = df.loc[df["s6"]!=1]
# df.drop(columns="s6", inplace=True)

# Find Examples

In [5]:
# collect used examples to exclude from test dataset
indexes_to_drop = []

In [6]:
# collect examples with single fault
examples = {}

for c in df.columns[1:]:
    if df[c].sum() == 0: continue
    examples[c] = {}
    i = df.loc[ (df[c] == 1) & (df[df.columns[1:][df.columns[1:]!=c]].sum(axis=1) == 0) ].sample(1)
    indexes_to_drop.append(i.index)
    i = i.values[0]
    examples[c]["req"] = i[0]
    examples[c]["vec"] = "[" + ",".join(map(str, i[1:])) + "]"

In [7]:
# collect examples with multiple faults
examples_multiple = {}

t = df.loc[df[df.columns[1:]].sum(axis=1) == 2]
idx = t.drop(columns="req").drop_duplicates().index
indexes_to_drop.append(idx)
t = df.iloc[idx, :].values

for r in t:
    c = "&".join(df.columns[1:][r[1:]==1])
    examples_multiple[c] = {}
    examples_multiple[c]["req"] = r[0]
    examples_multiple[c]["vec"] = "[" + ",".join(map(str, r[1:])) + "]"

In [8]:
# add both single and multiple into one place
examples.update(examples_multiple)

In [9]:
# join all examples in a text format to add to prompt
examples_txt = ""

for e in examples.values():
    examples_txt += f"Requirement: {e['req']}\n"
    examples_txt += f"Vector: {e['vec']}\n"
    examples_txt += "\n"

In [10]:
print("\n".join(examples_txt.split("\n")[:5]))

Requirement: The integrity of the communication link between the accelerator pedal sensors and the throttle control system must be ensured at all times
Vector: [1,0,0,0,0,0]

Requirement: If calibration issues arise, the system must inform the driver through the vehicle’s user interface immediately
Vector: [0,1,0,0,0,0]


In [11]:
# drop indexes used in examples
indexes_to_drop = np.concatenate(indexes_to_drop)
df.drop(index=indexes_to_drop, inplace=True)

# LLM

In [12]:
from prompts.SystemPrompts import SystemPrompt
from prompts.Sensors import Sensors
from prompts.UserPrompt import UserPrompt

In [13]:
# print(SystemPrompt.format(sensors=Sensors,examples=examples_txt))

In [14]:
# llm
llm = AzureChatOpenAI(
    deployment_name="gpt-35",
    temperature=0.0
)

# Run for all Requirements

In [67]:
def parse_result(res):
    '''Parse the LLM result'''
    pattern = r"\[(.*?)\]"
    vec = re.findall(pattern, res)[0]
    return f"[{vec}]".replace(" ", "")


def run_all_reqs(instance):
    # system prompt
    messages = [
        {'role': 'system',
        'content': SystemPrompt.format(sensors=Sensors,examples=examples_txt)}
    ]

    result = {}

    result["idx"] = instance[0]
    result["requirement"] = instance[1].iloc[0]
    result["true_vector"] = "[" + ",".join(map(str, instance[1].iloc[1:])) + "]"

    # add user prompt
    messages.append({"role":"user", "content":UserPrompt.format(req=result["requirement"])})

    # run LLM
    response = llm.invoke(messages)
    result["ai_response"] = response.content
    result["pred_vector"] = parse_result(result["ai_response"])

    result["accuracy"] = result["pred_vector"] == result["true_vector"]

    result["ai_token_usage"] = response.response_metadata["token_usage"]

    return result

In [68]:
results = [run_all_reqs(i) for i in tqdm.tqdm(df.iterrows())]

In [77]:
accuracy = 0
total_tokens = 0
total_completion_tokens = 0

for r in results:
    accuracy += r["accuracy"]
    total_tokens += r["ai_token_usage"]["total_tokens"]
    total_completion_tokens += r["ai_token_usage"]["completion_tokens"]

number_of_reqs = len(results)
accuracy /= len(results)
avg_token_per_req = total_tokens / len(results)
avg_completion_token_per_req = total_completion_tokens / len(results)

In [78]:
# save conversation and metadata
time = dt.now()

results_path = "results/conv_{model}_acc-{accuracy}_{time}.json"
results_path = results_path.format(model=llm.deployment_name,
                    time=time.strftime('%m.%d.%Y-%H:%M:%S'),
                    accuracy=round(accuracy, 3))

results_file = Path().cwd() / results_path
results_file.parent.mkdir(exist_ok=True)
results_file.touch()

with results_file.open("w") as f:
    json.dump({"accuracy": accuracy,
        "number_of_reqs": number_of_reqs,
        "total_tokens": total_tokens,
        "total_completion_tokens": total_completion_tokens,
        "avg_token_per_req": avg_token_per_req,
        "avg_completion_token_per_req": avg_completion_token_per_req,
        "responses": results}, f, indent=4)